In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import hypergeom


# Paths from your VS Code project
input_path = Path(
    r"C:\Users\imran\Desktop\RNAseq_capstone\data\raw\mart_kegg_enrichment_input_202611121443.csv"
)

output_path = Path(
    r"C:\Users\imran\Desktop\RNAseq_capstone\data\processed\mart_kegg_enrichment_results.csv"
)


def benjamini_hochberg(p_values):
    """Calculate BH-adjusted p-values, preserving the original order."""
    p_values = np.asarray(p_values, dtype=float)
    count = len(p_values)

    if count == 0:
        return p_values

    order = np.argsort(p_values)
    sorted_p_values = p_values[order]

    adjusted_sorted = (
        sorted_p_values
        * count
        / np.arange(1, count + 1)
    )

    # Enforce monotonic adjusted p-values from largest rank to smallest.
    adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
    adjusted_sorted = np.clip(adjusted_sorted, 0, 1)

    adjusted = np.empty(count, dtype=float)
    adjusted[order] = adjusted_sorted

    return adjusted


# Read identifiers as text so gene IDs stay intact.
df = pd.read_csv(
    input_path,
    encoding="utf-8-sig",
    dtype={
        "genotype": "string",
        "gene_id": "string",
        "pathway_key": "string",
        "kegg_pathway_id": "string",
        "kegg_pathway_name": "string",
    },
)

required_columns = {
    "genotype",
    "gene_id",
    "log2_fold_change",
    "padj",
    "pathway_key",
    "kegg_pathway_id",
    "kegg_pathway_name",
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        f"CSV is missing these required columns: {sorted(missing_columns)}"
    )

# Clean text and convert the statistical fields to numbers.
for column in [
    "genotype",
    "gene_id",
    "pathway_key",
    "kegg_pathway_id",
    "kegg_pathway_name",
]:
    df[column] = df[column].str.strip()

df["genotype"] = df["genotype"].str.upper()
df["log2_fold_change"] = pd.to_numeric(
    df["log2_fold_change"], errors="coerce"
)
df["padj"] = pd.to_numeric(df["padj"], errors="coerce")

# Keep tested genes with a usable fold change and a pathway link.
df = df.dropna(
    subset=[
        "genotype",
        "gene_id",
        "log2_fold_change",
        "padj",
        "pathway_key",
    ]
)

df = df[
    df["gene_id"].ne("")
    & df["pathway_key"].ne("")
].copy()

# Apply the project thresholds:
# adjusted p-value < 0.05 and absolute log2 fold change >= 1.
df["direction"] = np.select(
    [
        (df["padj"] < 0.05) & (df["log2_fold_change"] >= 1),
        (df["padj"] < 0.05) & (df["log2_fold_change"] <= -1),
    ],
    [
        "Up in nP",
        "Down in nP",
    ],
    default="Not DEG",
)

# Pathway names and IDs for the result table.
pathway_info = (
    df.groupby("pathway_key", as_index=False)
    .agg(
        kegg_pathway_id=("kegg_pathway_id", "first"),
        kegg_pathway_name=("kegg_pathway_name", "first"),
    )
)

all_results = []

for genotype in ["WT", "PDR2"]:
    genotype_df = df[df["genotype"] == genotype].copy()

    if genotype_df.empty:
        print(f"No eligible rows found for {genotype}; skipping.")
        continue

    # Background: distinct tested genes that have KEGG pathway annotations.
    background_genes = set(genotype_df["gene_id"].dropna().unique())
    background_size = len(background_genes)

    # Each pathway is represented by its set of distinct background genes.
    pathway_gene_sets = {
        pathway_key: set(group["gene_id"].dropna().unique())
        for pathway_key, group in genotype_df.groupby("pathway_key")
    }

    for direction in ["Up in nP", "Down in nP"]:
        foreground_genes = set(
            genotype_df.loc[
                genotype_df["direction"] == direction,
                "gene_id",
            ].dropna().unique()
        )
        foreground_size = len(foreground_genes)

        comparison_results = []

        for pathway_key, pathway_genes in pathway_gene_sets.items():
            pathway_size = len(pathway_genes)
            overlap_genes = pathway_genes & foreground_genes
            overlap_size = len(overlap_genes)

            if foreground_size == 0 or overlap_size == 0:
                p_value = 1.0
            else:
                # Upper-tail hypergeometric test:
                # probability of this many or more DEG genes in the pathway.
                p_value = float(
                    hypergeom.sf(
                        overlap_size - 1,
                        background_size,
                        pathway_size,
                        foreground_size,
                    )
                )

            comparison_results.append(
                {
                    "genotype": genotype,
                    "direction": direction,
                    "pathway_key": pathway_key,
                    "background_genes": background_size,
                    "foreground_genes": foreground_size,
                    "pathway_genes_in_background": pathway_size,
                    "foreground_genes_in_pathway": overlap_size,
                    "overlap_gene_ids": ";".join(sorted(overlap_genes)),
                    "p_value": p_value,
                }
            )

        # Adjust across all pathways for this genotype and direction.
        adjusted_values = benjamini_hochberg(
            [row["p_value"] for row in comparison_results]
        )

        for row, fdr_value in zip(comparison_results, adjusted_values):
            row["fdr_bh"] = fdr_value
            all_results.append(row)

if not all_results:
    raise ValueError(
        "No pathway results were calculated. Check that the CSV contains "
        "WT and PDR2 rows with pathway links."
    )

result_df = pd.DataFrame(all_results)

result_df = result_df.merge(
    pathway_info,
    on="pathway_key",
    how="left",
)

result_df = result_df.sort_values(
    ["genotype", "direction", "fdr_bh", "p_value"]
)

output_path.parent.mkdir(parents=True, exist_ok=True)
result_df.to_csv(output_path, index=False)

print(f"Saved pathway enrichment results to:\n{output_path}")
print(f"Pathway tests calculated: {len(result_df)}")

print("\nEligible annotated background and DEG counts:")
print(
    result_df[
        [
            "genotype",
            "direction",
            "background_genes",
            "foreground_genes",
        ]
    ]
    .drop_duplicates()
    .to_string(index=False)
)

Saved pathway enrichment results to:
C:\Users\imran\Desktop\RNAseq_capstone\data\processed\mart_kegg_enrichment_results.csv
Pathway tests calculated: 548

Eligible annotated background and DEG counts:
genotype  direction  background_genes  foreground_genes
    PDR2 Down in nP              4295               197
    PDR2   Up in nP              4295               283
      WT Down in nP              4095                17
      WT   Up in nP              4095                47
